In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import pickle

# Load the cleaned data.
df = pd.read_csv('data/disasters_clean.csv')

print("Data loaded!")
print("Shape:", df.shape)
df.head()

Data loaded!
Shape: (16126, 13)


,Year,Disaster Group,Disaster Subgroup,Disaster Type,Country,Continent,Region,Total Deaths,No Injured,No Affected,No Homeless,Total Affected,Total Damages ('000 US$)
0,1900,Natural,Climatological,Drought,Cabo Verde,Africa,Western Africa,11000.0,0.0,0.0,0.0,0.0,0.0
1,1900,Natural,Climatological,Drought,India,Asia,Southern Asia,1250000.0,0.0,0.0,0.0,0.0,0.0
2,1902,Natural,Geophysical,Earthquake,Guatemala,Americas,Central America,2000.0,0.0,0.0,0.0,0.0,25000.0
3,1902,Natural,Geophysical,Volcanic activity,Guatemala,Americas,Central America,1000.0,0.0,0.0,0.0,0.0,0.0
4,1902,Natural,Geophysical,Volcanic activity,Guatemala,Americas,Central America,6000.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Create a severity score.
df['severity_score'] = (
    np.log1p(df['Total Deaths']) +
    np.log1p(df['Total Affected'])
)

# Divide severity into 3 classes.
# Low = 0, Medium = 1, High = 2
df['severity_class'] = pd.cut(
    df['severity_score'],
    bins=3,
    labels=[0, 1, 2]
).astype(int)

print("Severity Classes:")
print(df['severity_class'].value_counts())

# Encode Disaster Type (convert text into numbers).
le = LabelEncoder()
df['Disaster_Type_Encoded'] = le.fit_transform(df['Disaster Type'])

print("\nDisaster Types Encoded:")
print(df[['Disaster Type', 'Disaster_Type_Encoded']].drop_duplicates())

Severity Classes:
severity_class
0    10665
1     5311
2      150
Name: count, dtype: int64

Disaster Types Encoded:
               Disaster Type  Disaster_Type_Encoded
0                    Drought                      1
2                 Earthquake                      2
3          Volcanic activity                     13
5        Mass movement (dry)                     11
7                      Storm                     12
12                     Flood                      5
16                  Epidemic                      3
25                 Landslide                     10
33                  Wildfire                     14
129     Extreme temperature                       4
209                      Fog                      6
890       Insect infestation                      9
12856                 Impact                      8
13456        Animal accident                      0
15814  Glacial lake outburst                      7


In [ ]:
# Define features and target variables.
X = df[['Year', 'Total Deaths', 'Total Affected', 
        "Total Damages ('000 US$)", 'Disaster_Type_Encoded']]
y = df['severity_class']

# Split the data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data size:", X_train.shape)
print("Testing data size:", X_test.shape)

# Build and train an XGBoost model.
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)
print("\n✅ Model training complete!")

# Check the accuracy.
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 Accuracy: {accuracy*100:.2f}%")

Training data size: (12900, 5)
Testing data size: (3226, 5)

✅ Model training complete!
🎯 Accuracy: 99.47%


In [ ]:
# Save the model.
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("✅ model.pkl saved!")

# Save the label encoder as well.
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("✅ label_encoder.pkl saved!")

✅ model.pkl saved!
✅ label_encoder.pkl saved!
